# Articulation

Requires `hyperdrone[examples]`. Run the three cells to save and display a 3-second `articulation.mp4`: an orange hinge rotates while its blue child slides. Coordinates are an angle in radians and a distance in asset-space metres; `s` scales the entire mechanism.


In [ ]:
import imageio.v2 as imageio
import numpy as np
from IPython.display import Video
from hyperdrone import render
from hyperdrone.camera import RGB

rod = render.primitives.box(size=(0.12, 1.6, 0.12), color=(1, 0.2, 0))
cube = render.primitives.box(size=(0.3, 0.3, 0.3), color=(0, 0.8, 1))
identity = render.make_transform()
offset = render.make_transform(position=(0, 0.5, 0))
base = render.make_transform(position=(2, 0, 0))
asset = render.ObjectAssembly()
pool = render.AssetPool()
scene = render.Scene()
scene.set_environment(background=(0.035, 0.045, 0.06))
renderer = render.Renderer(width=256, height=256, fidelity="low", num_probes=0,
                           num_overlays=1, max_overlay_instances=2, max_overlays_per_camera=1)


In [ ]:
hinge = asset.add_part(rod, identity)
block = asset.add_part(cube, offset, parent=hinge)
asset.set_joint(hinge, "revolute", axis=(1, 0, 0), coordinate=0)
asset.set_joint(block, "prismatic", axis=(0, 1, 0), coordinate=1)
handle = pool.add_assembly(asset)
renderer.init(scene, asset_pool=pool)
placement = renderer.spawn(0, handle, render.Pose(base))
renderer.attach(0, 0)
renderer.set_cameras(renderer.camera(position=(0, 0, 0), look_at=(1, 0, 0))[None])


In [ ]:
s = 1.0
frames = []
for phase in np.linspace(0, 2 * np.pi, 90, endpoint=False):
    angle, slide = phase, 0.3 * np.sin(2 * phase)
    renderer.set_pose(placement, render.Pose(base, scale=s), coordinates=[angle, slide])
    renderer.update()
    renderer.render()
    frames.append(renderer.outputs[RGB("rgba8")].read()[0, ..., :3].copy())
imageio.mimsave("articulation.mp4", frames, fps=30, codec="libx264")
Video("articulation.mp4", embed=True, html_attributes="controls loop")
